# TP03 — Arboles de decisión - Ejercicio 1
   Bases de Datos Masivas (11088) · Gustavo An Contardi - 182818
   

In [2]:
import math
import pandas as pd

# Ruta relativa: el notebook vive en TP03/ y los datos en TP03/data/
df = pd.read_csv("data/parcial.csv")

CLASE = "APRUEBA"
# Saco los atributos de las columnas en vez de escribirlos a mano.
# Este orden es el que usa max() para desempatar (decisión 4).
ATRIBUTOS = [c for c in df.columns if c != CLASE]
df

,ASISTENCIA,TPS,TRABAJA,MODALIDAD,APRUEBA
0,alta,completos,no,presencial,si
1,alta,parciales,si,virtual,si
2,alta,completos,si,presencial,si
3,alta,parciales,no,virtual,si
4,baja,completos,no,presencial,no
5,baja,parciales,si,presencial,no
6,baja,completos,si,virtual,no
7,baja,parciales,no,virtual,no
8,media,completos,si,presencial,si
9,media,completos,no,virtual,si


In [3]:
def entropia(etiquetas):
    """Entropía en bits de una columna de clase: H = -sum(p * log2 p)."""
    n = len(etiquetas)
    # value_counts solo devuelve las clases presentes, así que nunca
    # aparece p = 0 y no hay que tratar el log2(0) aparte.
    proporciones = etiquetas.value_counts() / n
    # Escribo -p*log2(p) adentro del sum para que un nodo puro dé 0.0
    # y no -0.0, que se imprime raro.
    return sum(-p * math.log2(p) for p in proporciones)

print(f"H(raíz) = {entropia(df[CLASE]):.4f}")

H(raíz) = 0.9852


In [4]:
def ganancia(df, atributo, clase):
    """Ganancia = H(nodo) - promedio ponderado de H en cada rama del atributo."""
    h_padre = entropia(df[clase])
    h_hijos = 0.0
    # groupby arma un subconjunto por cada valor del atributo
    for valor, sub in df.groupby(atributo):
        # Cada rama pesa según la fracción de filas que se lleva
        h_hijos += len(sub) / len(df) * entropia(sub[clase])
    return h_padre - h_hijos

for a in ATRIBUTOS:
    print(f"{a:<12} {ganancia(df, a, CLASE):.4f}")

ASISTENCIA   0.5917
TPS          0.0611
TRABAJA      0.0202
MODALIDAD    0.0000


In [5]:
def id3(df, clase, atributos, camino="raíz"):
    """Devuelve el árbol como dict anidado e imprime cómo decide en cada nodo.
    Hoja:  {"hoja": etiqueta, "conteo": {...}}
    Nodo:  {"atributo": nombre, "ramas": {valor: subárbol}}
    """
    conteo = df[clase].value_counts().to_dict()
    print(f"\nNodo [{camino}]  n={len(df)}  {conteo}  H={entropia(df[clase]):.4f}")

    # Corte 1: nodo puro, no hay nada más que separar
    if len(conteo) == 1:
        print("  -> hoja: nodo puro")
        return {"hoja": df[clase].iloc[0], "conteo": conteo}

    # Corte 2: se acabaron los atributos, decide la clase mayoritaria
    if not atributos:
        print("  -> hoja: no quedan atributos, decide la mayoría")
        return {"hoja": df[clase].mode()[0], "conteo": conteo}

    ganancias = {a: ganancia(df, a, clase) for a in atributos}
    for a, g in sorted(ganancias.items(), key=lambda x: -x[1]):
        print(f"  {a:<12} {g:.4f}")

    # Desempate: ante igual ganancia, max() devuelve el primero de la lista,
    # o sea el que aparece antes en las columnas del CSV.
    mejor = max(atributos, key=lambda a: ganancias[a])

    # Corte 3 (prepoda mínima): si ni el mejor atributo separa nada, no divido.
    # Comparo contra 1e-12 y no contra 0 por los errores de redondeo del float.
    if ganancias[mejor] <= 1e-12:
        print("  -> hoja: ningún atributo separa, decide la mayoría")
        return {"hoja": df[clase].mode()[0], "conteo": conteo}

    print(f"  -> divido por {mejor}")
    # ID3 no reutiliza un atributo categórico en la misma rama:
    # adentro de la rama ya vale lo mismo para todas las filas.
    restantes = [a for a in atributos if a != mejor]
    ramas = {}
    for valor, sub in df.groupby(mejor):
        ramas[valor] = id3(sub, clase, restantes, f"{camino} > {mejor}={valor}")
    return {"atributo": mejor, "ramas": ramas}

arbol = id3(df, CLASE, ATRIBUTOS)


Nodo [raíz]  n=14  {'si': 8, 'no': 6}  H=0.9852
  ASISTENCIA   0.5917
  TPS          0.0611
  TRABAJA      0.0202
  MODALIDAD    0.0000
  -> divido por ASISTENCIA

Nodo [raíz > ASISTENCIA=alta]  n=4  {'si': 4}  H=0.0000
  -> hoja: nodo puro

Nodo [raíz > ASISTENCIA=baja]  n=4  {'no': 4}  H=0.0000
  -> hoja: nodo puro

Nodo [raíz > ASISTENCIA=media]  n=6  {'si': 4, 'no': 2}  H=0.9183
  TPS          0.4591
  TRABAJA      0.2516
  MODALIDAD    0.0000
  -> divido por TPS

Nodo [raíz > ASISTENCIA=media > TPS=completos]  n=3  {'si': 3}  H=0.0000
  -> hoja: nodo puro

Nodo [raíz > ASISTENCIA=media > TPS=parciales]  n=3  {'no': 2, 'si': 1}  H=0.9183
  TRABAJA      0.9183
  MODALIDAD    0.2516
  -> divido por TRABAJA

Nodo [raíz > ASISTENCIA=media > TPS=parciales > TRABAJA=no]  n=1  {'si': 1}  H=0.0000
  -> hoja: nodo puro

Nodo [raíz > ASISTENCIA=media > TPS=parciales > TRABAJA=si]  n=2  {'no': 2}  H=0.0000
  -> hoja: nodo puro


In [6]:
def clasificar(nodo, fila):
    # Baja por el árbol siguiendo el valor de la fila hasta llegar a una hoja
    while "hoja" not in nodo:
        nodo = nodo["ramas"][fila[nodo["atributo"]]]
    return nodo["hoja"]

pred = df.apply(lambda fila: clasificar(arbol, fila), axis=1)
print(f"Aciertos sobre las 14 filas: {(pred == df[CLASE]).sum()} de {len(df)}")

Aciertos sobre las 14 filas: 14 de 14


## Conclusiones

### Cómo lo implementé

Usé pandas para cargar y filtrar los datos. La entropía la escribí a mano con
`math.log2`, para que se vea la fórmula. La función de ganancia usa la de entropía
sobre el nodo y sobre cada rama, y resta el promedio ponderado.

El árbol lo arma una función recursiva (ID3) que en cada nodo imprime la ganancia de
cada atributo y cuál elige. Corta en tres casos: cuando el nodo es puro, cuando no
quedan atributos y cuando ningún atributo tiene ganancia mayor a 0. Para los empates
definí que gane el primer atributo según el orden de las columnas. Al final no hizo
falta: en ningún nodo hubo empate.

### Árbol obtenido

```
ASISTENCIA = alta  ->  si
ASISTENCIA = baja  ->  no
ASISTENCIA = media
    TPS = completos  ->  si
    TPS = parciales
        TRABAJA = no  ->  si
        TRABAJA = si  ->  no
```

### Qué veo en el árbol

En la raíz gana ASISTENCIA, con ganancia 0,5917 contra 0,0611 de TPS. Gana por mucho
porque con `alta` y `baja` las ramas quedan puras: todos los de asistencia alta
aprueban y todos los de baja no. Solo queda mezclada la rama `media`.

Dentro de `media` se divide por TPS (0,4591) y, en los que tienen TPS parciales, por
TRABAJA (0,9183).

MODALIDAD no aparece en el árbol. Su ganancia es 0 en la raíz y en `media`: dividir
por modalidad no cambia la proporción de aprobados. En el último nodo da 0,2516, pero
pierde contra TRABAJA.

El árbol clasifica bien las 14 filas, pero las mide sobre los mismos datos con los que
lo armé, así que eso no dice cómo le iría con datos nuevos. Además, la hoja
`TRABAJA = no` decide con una sola fila. El árbol crece hasta que todas las hojas son
puras, y con tan pocos datos algunas reglas se apoyan en uno o dos casos. No le
tendría mucha confianza a esa hoja.